# 📘 Tutorial 8: Joining Metadata with Measurements

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

Measurement tables often contain only sample IDs and values. Metadata tables explain what those sample IDs mean. Joining them correctly is essential before grouping, colouring, faceting, or comparing conditions.

---

## 🎯 Learning objectives

By the end of this notebook, you should be able to:

- recognise metadata tables,
- use `merge`,
- join sample IDs with conditions,
- validate joins,
- detect duplicated or unmatched rows,
- explain why metadata matters for plotting and analysis.


## 🔧 Setup


In [ ]:
from pathlib import Path

import pandas as pd


## 1. Read measurements and metadata


In [ ]:
project_root = Path.cwd()
if not (project_root / "data" / "part1").exists():
    project_root = project_root.parent

data_dir = project_root / "data" / "part1"
measurements = pd.read_csv(data_dir / "replicate_measurements.csv")
metadata = pd.read_csv(data_dir / "sample_metadata.csv")


In [ ]:
measurements.head()


In [ ]:
metadata


The measurements contain repeated response values. The metadata describes each sample.


## 2. Check the join key


In [ ]:
print("Measurement sample IDs:", sorted(measurements["sample_id"].unique()))
print("Metadata sample IDs:", sorted(metadata["sample_id"].unique()))


In [ ]:
metadata["sample_id"].duplicated().any()


For this join, each sample ID should appear once in the metadata table. Duplicate metadata keys can accidentally duplicate measurement rows.


## 3. Merge metadata onto measurements


In [ ]:
measurement_values = measurements.drop(columns=["condition"])

merged = measurement_values.merge(
    metadata,
    on="sample_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

merged


`validate="many_to_one"` states our expectation: many measurement rows can match one metadata row.


## 4. Check for unmatched rows


In [ ]:
merged["_merge"].value_counts()


In [ ]:
merged.loc[merged["_merge"] != "both"]


There are no unmatched rows here, so we can remove the diagnostic merge indicator from the clean result.


In [ ]:
merged_clean = merged.drop(columns="_merge")
merged_clean.head()


## 5. Demonstrate an unmatched sample


In [ ]:
extra_measurement = pd.DataFrame(
    {
        "sample_id": ["S99"],
        "replicate": [1],
        "time_min": [10],
        "response": [0.50],
    }
)

with_unmatched = pd.concat([measurement_values, extra_measurement], ignore_index=True)

unmatched_demo = with_unmatched.merge(
    metadata,
    on="sample_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

unmatched_demo.loc[unmatched_demo["_merge"] != "both"]


The unmatched row is visible because the merge indicator was kept during inspection.


## 6. Use metadata for a summary


In [ ]:
summary = (
    merged_clean.groupby(["condition", "concentration_mM"], as_index=False)
    .agg(mean_response=("response", "mean"), n=("response", "count"))
    .sort_values("concentration_mM")
)

summary


The measurement table alone had sample IDs. The joined table can now support condition-aware plotting.


## 7. Checkpoint: validate before trusting

Before using a merged table for plotting, check:

- whether the join key is unique on the metadata side,
- whether row counts changed as expected,
- whether any rows are `left_only` or `right_only`,
- whether duplicated keys created duplicated measurements,
- whether the joined columns have the expected missingness.


## 🧭 Key takeaways

- Metadata gives measurement rows analytical meaning.
- `merge` joins tables using shared keys such as `sample_id`.
- `validate` documents the expected relationship between tables.
- Merge indicators help detect unmatched rows.
- Joined metadata is often what makes grouped plotting possible.
